***Example  — Scaling Before Split (Leakage)***

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Load data
data = pd.read_csv("placement.csv")

# Features and target
X = data[['CGPA']]
y = data['Package_LPA']

# DATA LEAKAGE
# WRONG: scaler sees full dataset

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# Split after scaling
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

# Model
model = LinearRegression()

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Score
print("R2 Score:", r2_score(y_test, y_pred))

R2 Score: 0.9925617081750644


***Correct Approach (No Leakage)***

In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Load data
data = pd.read_csv("placement.csv")

# Features and target
X = data[['CGPA']]
y = data['Package_LPA']

# Split FIRST
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Correct scaling
# Fit ONLY on training data

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

# Only transform test data
X_test_scaled = scaler.transform(X_test)

# Model
model = LinearRegression()

# Train
model.fit(X_train_scaled, y_train)

# Predict
y_pred = model.predict(X_test_scaled)

# Score
print("R2 Score:", r2_score(y_test, y_pred))

R2 Score: 0.9925617081750644


In [ ]:
'''
Leakage-Prone Operations

Never fit these on full data before split:
✔ StandardScaler
✔ MinMaxScaler
✔ PCA
✔ Imputer
✔ Feature Selection
✔ Encoding
'''

In [3]:
# PIPELINE EXAMPLE

# Problem:
# Predict Package_LPA using CGPA

# Pipeline Flow:
#
# Missing Values
#       ↓
# Scaling
#       ↓
# Model Training
#       ↓
# Prediction

# Benefits:
# ✔ No Data Leakage
# ✔ Clean Code
# ✔ Works perfectly with CV/GridSearchCV
# ✔ Production Ready

# ======================================

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

# ======================================
# LOAD DATA
# ======================================

data = pd.read_csv("placement.csv")

# ======================================
# FEATURES AND TARGET
# ======================================

X = data[['CGPA']]
y = data['Package_LPA']

# ======================================
# TRAIN TEST SPLIT
# ======================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ======================================
# BUILD PIPELINE
# ======================================

pipe = Pipeline([

    # STEP 1:
    # Fill missing values using mean

    ('imputer', SimpleImputer(strategy='mean')),

    # STEP 2:
    # Standardize features

    ('scaler', StandardScaler()),

    # STEP 3:
    # Train model

    ('model', LinearRegression())

])

# ======================================
# TRAIN PIPELINE
# ======================================

pipe.fit(X_train, y_train)

# ======================================
# PREDICTIONS
# ======================================

y_pred = pipe.predict(X_test)

# ======================================
# EVALUATION
# ======================================

print("R2 Score:")
print(r2_score(y_test, y_pred))

print()

print("MSE:")
print(mean_squared_error(y_test, y_pred))

print()

print("MAE:")
print(mean_absolute_error(y_test, y_pred))

# ======================================
# INTERNAL WORKING OF PIPELINE
# ======================================

# During fit():
#
# 1. imputer.fit(X_train)
# 2. imputer.transform(X_train)
#
# 3. scaler.fit(X_train)
# 4. scaler.transform(X_train)
#
# 5. model.fit()

# During predict():
#
# 1. imputer.transform(X_test)
# 2. scaler.transform(X_test)
#
# 3. model.predict()

# IMPORTANT:
# Test data NEVER participates in fit()

# Therefore:
# ✔ No Data Leakage

# ======================================
# IMPORTANT INTERVIEW POINTS
# ======================================

# ✔ Pipeline automates preprocessing
# ✔ Prevents train-test contamination
# ✔ Makes deployment easier
# ✔ Works safely with Cross Validation
# ✔ Keeps code modular and clean

# ======================================
# MOST IMPORTANT LINE
# ======================================

# Pipeline ensures all preprocessing
# happens ONLY on training data.

R2 Score:
0.9925617081750644

MSE:
0.009962546850236516

MAE:
0.0862766893465221
